In [2]:
import pandas as pd
import numpy as np

# 1. Cargar los datos
# Usamos encoding='latin1' y sep=';' porque es el formato original de tu archivo
df = pd.read_csv('esterilizacion-de-caninos-y-felinos-valle-dell-cauca.csv', encoding='latin1', sep=';')

print("--- Dimensiones originales ---")
print(df.shape)

# 2. Limpieza de Nombres de Columnas
# Pasamos todo a minúsculas, quitamos espacios al inicio/final y reemplazamos espacios internos por guiones bajos
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# 3. Limpieza de Variables Categóricas (Inconsistencias de digitación)
# Estandarizamos "Canino", "Caninoss" a "Caninos"
df['tipo_de_animal'] = df['tipo_de_animal'].replace({
    'Canino': 'Caninos',
    'Caninoss': 'Caninos'
})

# Estandarizamos el texto de los municipios (quitar espacios en blanco adicionales)
df['municipio'] = df['municipio'].str.strip().str.title()
df['lugar_del_municipio'] = df['lugar_del_municipio'].str.strip()

# 4. Limpieza de Fechas
# Convertimos la columna de texto a formato de fecha real (datetime)
df['fecha_de_esterilización'] = pd.to_datetime(df['fecha_de_esterilización'], format='%d/%m/%Y', errors='coerce')

# 5. Tratamiento de Duplicados
# Eliminamos filas que sean exactamente iguales (si alguien registró lo mismo dos veces por error)
df = df.drop_duplicates()

# 6. Limpieza de Coordenadas (Latitud y Longitud)
# Tus coordenadas tienen un error de digitación con múltiples puntos (ej. 3.454.710...)
# Esta función limpia ese formato dejando solo el primer separador decimal
def clean_coordinates(coord):
    if pd.isna(coord): return np.nan
    coord = str(coord).replace('.', '') # Quitamos todos los puntos
    # Como el Valle del Cauca está en Lat: ~3-4, Lon: ~-76, ponemos el punto en el lugar correcto
    if coord.startswith('-'):
        return float(coord[:3] + '.' + coord[3:])
    else:
        return float(coord[:1] + '.' + coord[1:])

df['latitud_limpia'] = df['latitud'].apply(clean_coordinates)
df['longitud_limpia'] = df['longitud'].apply(clean_coordinates)

# Opcional: Eliminar las columnas originales de coordenadas defectuosas
df = df.drop(columns=['latitud', 'longitud'])

# 7. Valores Nulos
# Verificamos si quedaron datos vacíos
print("\n--- Valores nulos por columna ---")
print(df.isnull().sum())

# 8. Guardar el archivo limpio
# Lo guardamos como un CSV estándar, con separador de comas (,) y encoding universal (utf-8)
df.to_csv('esterilizaciones_valle_limpio.csv', index=False, encoding='utf-8')

print("\n--- ¡Limpieza completada! Archivo 'esterilizaciones_valle_limpio.csv' generado ---")
print(df.head())

--- Dimensiones originales ---
(384, 11)

--- Valores nulos por columna ---
fecha_de_esterilización    0
codigo_departamento        0
departamento               0
codigo_municipio           0
municipio                  0
lugar_del_municipio        0
sexo                       0
tipo_de_animal             0
total_esterizaciones       0
latitud_limpia             0
longitud_limpia            0
dtype: int64

--- ¡Limpieza completada! Archivo 'esterilizaciones_valle_limpio.csv' generado ---
  fecha_de_esterilización  codigo_departamento     departamento  \
0              2024-07-13                   76  Valle del Cauca   
1              2024-07-13                   76  Valle del Cauca   
2              2024-07-13                   76  Valle del Cauca   
3              2024-07-13                   76  Valle del Cauca   
4              2024-08-10                   76  Valle del Cauca   

   codigo_municipio municipio             lugar_del_municipio     sexo  \
0             76497    Obando  